# AD GWAS loci: SuSiE-RSS, SuSiE-rRSS and the other QC methods (data processing)

For every study × region run in `29.FigS1.AD_GWAS_QC_Methods`, the locus data of the multi-CS plots (Fig 6D: AD GWAS −log10 P, stored SuSiE-RSS and SuSiE-rRSS fits, cV2F, xQTL inclusion and colocalization, 2026 AD GWAS colocalization) plus the PIPs and credible sets of CARMA, DENTIST, DENTIST + RAISS, SLALOM + RAISS and RSparsePro (`load_qc_method_pips()` in `R/utils.R`; positions from `<unit>.inputs.rds`). Methods that failed in `29` are skipped with a message.

Units: every study × region run in `29` (`ADGWAS_QC_methods/result/`): the five Fig S1 regions, every study × region whose SuSiE-RSS and SuSiE-rRSS credible sets differ (`units_differ.tsv` from `[qc_units]`), and the SER fallback analyses not among them (`units_fallback.tsv` from `[qc_units_fallback]`); APOE excluded. The APOE block (all methods, `ADGWAS_QC_methods_APOE/result/`) is kept separately: a credible set count table and the same locus data, one locus per study (last two sections).

**Outputs:**
- `figure_data/AD_GWAS_QC_Methods_Loci.rds` — named list of loci (same structure as Fig 6D, more methods in `pip_df`)
- `figure_data/AD_GWAS_QC_Methods_Loci_CS_Counts.csv` — credible sets per unit and method, with whether the unit is a differing region and whether SuSiE-rRSS used the SER fallback
- `figure_data/AD_GWAS_QC_Methods_APOE_CS_Counts.csv` — credible sets per study and method at the APOE block
- `figure_data/AD_GWAS_QC_Methods_APOE_Loci.rds` — APOE block, one locus per study (same structure as `AD_GWAS_QC_Methods_Loci.rds`)

In [1]:
source("../../R/utils.R")
library(pecotmr)
devtools::load_all("/hpc/users/sunh17/haochen/rss_qc/pecotmr_latest/pecotmr", compile = FALSE)   # getSumStatsDf()
library(readxl)
filter <- dplyr::filter; select <- dplyr::select

data_name <- "AD_GWAS_QC_Methods_Loci"
qc_root   <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods/result"   # notebook 29

ℹ Loading pecotmr
Creating a generic function for ‘endoapply’ from package ‘S4Vectors’ in package ‘pecotmr’

Warning message:
“Failed to load at least one DLL.
Caused by error in `getDLLRegisteredRoutines.DLLInfo()`:
! must specify DLL via a “DLLInfo” object. See getLoadedDLLs()”


In [2]:
### AD xQTL evidence and 2026 AD GWAS colocalization, as in Fig 6E
AD_loci <- suppressMessages(suppressWarnings(
  read_excel("/hpc/users/sunh17/haochen/rss_qc/2026_output/ADGWAS_finemap/unified_AD_loci_xQTL_summary.xlsx", skip = 2)))
transmap_long <- tidy_transmap(read_tsv(
  "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_finemap/AD_GWAS_EUR_rss_L10/AD_GWAS_rss_ebmix_r4_L10/export/transmap_AD_target_results_pv1e4.tsv",
  show_col_types = FALSE))

In [5]:
## ---- one locus per "<study>.<block_id>" run in notebook 29 -----------------------------
units <- basename(list.dirs(qc_root, recursive = FALSE))
units

loci <- map(setNames(units, units), function(u) {
  sid <- str_split_fixed(u, "\\.", 2)[, 1]; bid <- str_split_fixed(u, "\\.", 2)[, 2]
  loc <- load_locus(bid, sid, AD_loci)                           # stored SuSiE-RSS / SuSiE-rRSS fits, GWAS, xQTL
  loc$tm <- locus_transmap(transmap_long, loc$reg)
  loc$pip_df <- bind_rows(loc$pip_df, load_qc_method_pips(u, qc_root))   # CARMA, DENTIST, ..., RSparsePro
  loc
})

## methods and credible-set counts per locus; differing regions from 29 [qc_units]
differ <- read_tsv(file.path(dirname(qc_root), "units", "units_differ.tsv"), show_col_types = FALSE)
## SER fallback analyses from 29 [qc_units_fallback] (all flagged units, with or without a differing region)
fb_file  <- file.path(dirname(qc_root), "units", "units_fallback.tsv")
fallback <- if (file.exists(fb_file)) read_tsv(fb_file, show_col_types = FALSE) else tibble(unit = character())
cs_counts <- map_dfr(loci, ~ .x$pip_df %>% group_by(method) %>%
                       summarise(n_variants = n(), n_cs = n_distinct(cs[cs != "0"]), .groups = "drop"), .id = "unit") %>%
  mutate(differing_region = unit %in% differ$unit, ser_fallback = unit %in% fallback$unit)
cs_counts %>% select(unit, method, n_cs) %>% pivot_wider(names_from = method, values_from = n_cs) %>% as.data.frame()
write_csv(cs_counts, file.path(data_dir, paste0(data_name, "_CS_Counts.csv")))

save_data(loci, data_name)

[1] "AD_Bellenguez_2022.chr11_119829118_122612025"                     
 [2] "AD_Bellenguez_2022.chr11_84267999_86714492"                       
 [3] "AD_Bellenguez_2022.chr17_45383525_50162864"                       
 [4] "AD_Bellenguez_2022.chr19_90949_1507536"                           
 [5] "AD_Bellenguez_2022.chr5_86472045_89595713"                        
 [6] "AD_Bellenguez_2022.chr6_40377803_42070711"                        
 [7] "AD_Bellenguez_2022.chr7_98297733_101733715"                       
 [8] "AD_Bellenguez_2022.chr8_26225312_29469590"                        
 [9] "AD_Bellenguez_EADB_2022.chr17_45383525_50162864"                  
[10] "AD_Bellenguez_EADB_2022.chr2_231326994_235132526"                 
[11] "AD_Bellenguez_main_2026.chr11_84267999_86714492"                  
[12] "AD_Bellenguez_main_2026.chr15_58149891_59314387"                  
[13] "AD_Bellenguez_main_2026.chr15_61125463_66680537"                  
[14] "AD_Bellenguez_main_2026.chr16_70662193_72901302"                  
[15] "AD_Bellenguez_main_2026.chr16_81293081_82644764"                  
[16] "AD_Bellenguez_main_2026.chr17_1013206_2799513"                    
[17] "AD_Bellenguez_main_2026.chr17_4611485_5782462"                    
[18] "AD_Bellenguez_main_2026.chr19_31660546_33534775"                  
[19] "AD_Bellenguez_main_2026.chr19_90949_1507536"                      
[20] "AD_Bellenguez_main_2026.chr2_105444433_106385118"                 
[21] "AD_Bellenguez_main_2026.chr2_125689597_127728648"                 
[22] "AD_Bellenguez_main_2026.chr6_40377803_42070711"                   
[23] "AD_Bellenguez_main_2026.chr8_26225312_29469590"                   
[24] "AD_Bellenguez_nobiobank_2026.chr11_119829118_122612025"           
[25] "AD_Bellenguez_nobiobank_2026.chr11_56858541_63818332"             
[26] "AD_Bellenguez_nobiobank_2026.chr15_61125463_66680537"             
[27] "AD_Bellenguez_nobiobank_2026.chr17_60570445_65149278"             
[28] "AD_Bellenguez_nobiobank_2026.chr19_90949_1507536"                 
[29] "AD_Bellenguez_nobiobank_2026.chr3_154360351_156292136"            
[30] "AD_Bellenguez_nobiobank_2026.chr5_86472045_89595713"              
[31] "AD_Bellenguez_nobiobank_2026.chr8_26225312_29469590"              
[32] "AD_Bellenguez_noproxy_2026.chr11_119829118_122612025"             
[33] "AD_Bellenguez_noproxy_2026.chr11_84267999_86714492"               
[34] "AD_Bellenguez_noproxy_2026.chr12_122044856_124768254"             
[35] "AD_Bellenguez_noproxy_2026.chr15_58149891_59314387"               
[36] "AD_Bellenguez_noproxy_2026.chr15_61125463_66680537"               
[37] "AD_Bellenguez_noproxy_2026.chr17_45383525_50162864"               
[38] "AD_Bellenguez_noproxy_2026.chr19_31660546_33534775"               
[39] "AD_Bellenguez_noproxy_2026.chr19_90949_1507536"                   
[40] "AD_Bellenguez_noproxy_2026.chr21_24215031_26405838"               
[41] "AD_Bellenguez_noproxy_2026.chr6_45129743_49919132"                
[42] "AD_Bellenguez_noproxy_2026.chr8_26225312_29469590"                
[43] "AD_Jansen_2021.chr14_92293029_94303139"                           
[44] "AD_Kunkle_Stage1_2019.chr2_125689597_127728648"                   
[45] "AD_Wightman_Excluding23andMe_2021.chr1_205972031_208461272"       
[46] "AD_Wightman_Excluding23andMe_2021.chr11_56858541_63818332"        
[47] "AD_Wightman_Excluding23andMe_2021.chr14_92293029_94303139"        
[48] "AD_Wightman_Excluding23andMe_2021.chr19_53788025_55215821"        
[49] "AD_Wightman_Excluding23andMe_2021.chr19_90949_1507536"            
[50] "AD_Wightman_Excluding23andMe_2021.chr2_87936212_92062586"         
[51] "AD_Wightman_Excluding23andMe_2021.chr7_98297733_101733715"        
[52] "AD_Wightman_Excluding23andMe_2021.chr8_143034200_145075767"       
[53] "AD_Wightman_ExcludingUKBand23andME_2021.chr11_119829118_122612025"
[54] "AD_Wightman_ExcludingUKBand23andME_2021.chr11_56858541_63818332"  
[55] "AD_Wightman_ExcludingUKBand23andME_2021.chr16_6839898

unit,CARMA,DENTIST,DENTIST + RAISS,RSparsePro,SLALOM + RAISS,SuSiE-RSS,SuSiE-rRSS
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
AD_Bellenguez_2022.chr11_119829118_122612025,2,2,2,1,2,2,2
AD_Bellenguez_2022.chr11_84267999_86714492,2,2,2,1,2,2,1
AD_Bellenguez_2022.chr17_45383525_50162864,2,1,1,2,1,1,0
AD_Bellenguez_2022.chr19_90949_1507536,1,2,2,1,2,2,2
AD_Bellenguez_2022.chr5_86472045_89595713,1,1,1,1,1,1,1
AD_Bellenguez_2022.chr6_40377803_42070711,2,2,2,3,2,2,2
AD_Bellenguez_2022.chr7_98297733_101733715,1,1,1,1,1,1,1
AD_Bellenguez_2022.chr8_26225312_29469590,2,2,2,2,2,2,2
AD_Bellenguez_EADB_2022.chr17_45383525_50162864,1,1,1,1,1,1,1


In [10]:
cs_counts

unit,method,n_variants,n_cs,differing_region,ser_fallback
<chr>,<chr>,<int>,<int>,<lgl>,<lgl>
AD_Bellenguez_2022.chr11_119829118_122612025,CARMA,7783,2,TRUE,FALSE
AD_Bellenguez_2022.chr11_119829118_122612025,DENTIST,7783,2,TRUE,FALSE
AD_Bellenguez_2022.chr11_119829118_122612025,DENTIST + RAISS,7783,2,TRUE,FALSE
AD_Bellenguez_2022.chr11_119829118_122612025,RSparsePro,7783,1,TRUE,FALSE
AD_Bellenguez_2022.chr11_119829118_122612025,SLALOM + RAISS,7783,2,TRUE,FALSE
AD_Bellenguez_2022.chr11_119829118_122612025,SuSiE-RSS,7783,2,TRUE,FALSE
AD_Bellenguez_2022.chr11_119829118_122612025,SuSiE-rRSS,7783,2,TRUE,FALSE
AD_Bellenguez_2022.chr11_84267999_86714492,CARMA,7163,2,TRUE,FALSE
AD_Bellenguez_2022.chr11_84267999_86714492,DENTIST,7163,2,TRUE,FALSE


## APOE block: credible set counts of every method

In [4]:
`%|NA|%` <- function(a, b) if (is.na(a)) b else a
## ---- APOE block: credible sets per study and method (29 APOE runs) ----------------------
apoe_root <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods_APOE/result"
apoe_cs <- map_dfr(list.files(apoe_root, pattern = "\\.rds$", recursive = TRUE, full.names = TRUE) %>%
                     discard(~ grepl("\\.inputs\\.rds$", .x)), function(f) {
  r <- readRDS(f)
  labs <- c(qc_method_labels, susie_rss = "SuSiE-RSS", susie_rrss = "SuSiE-rRSS")
  tibble(study = r$study, method = unname(labs[r$method]) %|NA|% r$method,
         n_cs = if (is.null(r$error)) length(r$cs_variants %||% r$fit$sets$cs) else NA_integer_,
         ser_fallback = r$ser_fallback %||% NA, error = r$error %||% NA_character_)
})
apoe_cs %>% select(study, method, n_cs) %>% pivot_wider(names_from = method, values_from = n_cs) %>% as.data.frame()
filter(apoe_cs, !is.na(error)) %>% select(study, method, error)            # failed runs, if any
write_csv(apoe_cs, file.path(data_dir, "AD_GWAS_QC_Methods_APOE_CS_Counts.csv"))

study,CARMA,DENTIST,DENTIST + RAISS,RSparsePro,SLALOM + RAISS,SuSiE-rRSS,SuSiE-RSS
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
AD_Bellenguez_2022,9,7,4,7,9,1,9
AD_Bellenguez_EADB_2022,6,10,4,3,7,1,3
AD_Bellenguez_EADI_2022,1,1,7,1,1,1,1
AD_Bellenguez_GRACE_2022,1,1,3,1,2,1,2
AD_Bellenguez_main_2026,10,6,3,10,10,1,9
AD_Bellenguez_nobiobank_2026,7,9,5,7,9,1,9
AD_Bellenguez_noproxy_2026,NA,4,4,10,10,1,10
AD_Jansen_2021,10,10,9,1,9,1,9
AD_Kunkle_Stage1_2019,7,10,7,1,10,1,10


study,method,error
<chr>,<chr>,<chr>


## APOE block: locus data of every method, one locus per study

In [11]:
## ---- APOE block: same locus data as the other loci (stored SuSiE-RSS / SuSiE-rRSS fits + 29 APOE runs) ----
apoe_root  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/ADGWAS_QC_methods_APOE/result"
apoe_units <- basename(list.dirs(apoe_root, recursive = FALSE))

apoe_loci <- map(setNames(apoe_units, apoe_units), function(u) {
  sid <- str_split_fixed(u, "\\.", 2)[, 1]; bid <- str_split_fixed(u, "\\.", 2)[, 2]
  tryCatch({
    loc <- load_locus(bid, sid, AD_loci)                           # stored SuSiE-RSS / SuSiE-rRSS fits, GWAS, xQTL
    loc$tm <- locus_transmap(transmap_long, loc$reg)
    loc$pip_df <- bind_rows(loc$pip_df, load_qc_method_pips(u, apoe_root))   # CARMA, DENTIST, ..., RSparsePro
    loc
  }, error = function(e) { message(u, ": ", conditionMessage(e)); NULL })
}) %>% compact()

## credible sets per study and method in the saved loci
map_dfr(apoe_loci, ~ .x$pip_df %>% group_by(method) %>% summarise(n_cs = n_distinct(cs[cs != "0"]), .groups = "drop"),
        .id = "unit") %>%
  pivot_wider(names_from = method, values_from = n_cs) %>% as.data.frame()

save_data(apoe_loci, "AD_GWAS_QC_Methods_APOE_Loci")

unit,CARMA,DENTIST,DENTIST + RAISS,RSparsePro,SLALOM + RAISS,SuSiE-RSS,SuSiE-rRSS
<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
AD_Bellenguez_2022.chr19_42346101_46842901,9,7,4,7,9,5,1
AD_Bellenguez_EADB_2022.chr19_42346101_46842901,6,10,4,3,7,2,1
AD_Bellenguez_EADI_2022.chr19_42346101_46842901,1,1,7,1,1,1,1
AD_Bellenguez_GRACE_2022.chr19_42346101_46842901,1,1,3,1,2,1,1
AD_Bellenguez_main_2026.chr19_42346101_46842901,10,6,3,10,10,7,1
AD_Bellenguez_nobiobank_2026.chr19_42346101_46842901,7,9,5,7,9,2,1
AD_Bellenguez_noproxy_2026.chr19_42346101_46842901,NA,4,4,10,10,7,1
AD_Jansen_2021.chr19_42346101_46842901,10,10,9,1,9,9,0
AD_Kunkle_Stage1_2019.chr19_42346101_46842901,7,10,5,1,10,9,1
